# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² mass spectrometry dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best FAIR data practices.

### Dataset Source
The dataset source is provided via a Croissant schema at:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will enumerate all record sets, display their `@id`s, and show field and column `@id`s for each. This helps you reference and select the data to extract and analyze.

In [ ]:
# List all record sets with their @id, fields, and columns
print("Available Record Sets and their Fields/Columns:")

record_sets = metadata.record_sets
if record_sets:
    for rs in record_sets:
        print(f"\nRecord Set: {rs.id}")
        print(f"  Name: {getattr(rs, 'name', 'N/A')}")
        if hasattr(rs, 'fields') and rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - @id: {field.id}, name: {getattr(field, 'name', 'N/A')}")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - @id: {col.id}, name: {getattr(col, 'name', 'N/A')}")
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All entities (record sets, fields, and columns) are referenced by their `@id` in Croissant.

We'll demonstrate extraction for each record set (if more than one is found).

In [ ]:
# Choose record set(s) by their @id(s) from those discovered above.
# If no record sets, you may instantiate record_sets manually by @id as published.

# For demonstration, auto-populate record_set_ids by @id:
record_sets = metadata.record_sets
record_set_ids = [rs.id for rs in record_sets] if record_sets else []

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nExtracting data from record set: {record_set_id}")
    records = list(ds.records(record_set=record_set_id))
    print(f"Loaded {len(records)} records.")
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Fields/columns in this set: {df.columns.tolist()}")
    display(df.head())

# For the next step, select one record set for deeper analysis:
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nProceeding with record set: {main_record_set_id}")
else:
    main_record_set_id = None
    print("No record sets available for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing numeric fields, and grouping by key attributes.

_All field and column references use their Croissant `@id`._

In [ ]:
# Ensure a record set and DataFrame are available
import numpy as np

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]

    # Find likely numeric columns/fields by dtype or name, fallback if needed
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_candidates:
        # Attempt to guess common numeric fields
        numeric_candidates = [col for col in df.columns if 'count' in col.lower() or 'abundance' in col.lower() or 'mw' in col.lower() or 'score' in col.lower() or 'percent' in col.lower()]

    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field for EDA: {numeric_field_id}")
        # Ensure column is numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].mean() if not np.isnan(df[numeric_field_id].mean()) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a likely categorical field
        group_candidates = [col for col in df.columns if col != numeric_field_id and df[col].dtype == object]
        group_field_id = group_candidates[0] if group_candidates else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
        else:
            print("No categorical field suitable for grouping found.")
    else:
        print("No numeric field detected in the records set.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between numeric and categorical fields in the dataset.

Below, we plot histograms for the chosen numeric field and a barplot for group means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and 'numeric_field_id' in locals():
    df = dataframes[main_record_set_id]
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True, color='teal')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # Grouped means bar plot
    if 'grouped_df' in locals() and group_field_id:
        plt.figure(figsize=(10,5))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, color='coral')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated:
- Loading FAIR² dataset metadata and records directly from a Croissant schema via `mlcroissant`.
- Enumerating all record sets, fields, and their `@id`s for reproducible reference.
- Data extraction and preliminary EDA with selection, filtering, normalization, and group-by by field `@id`s.
- Basic visualization of numeric distributions and grouped means.

For further analysis, refer to [mlcroissant documentation](https://mlcommons.github.io/croissant/api_reference.html) and the dataset's own [schema URL](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).